In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [5]:
from datetime import datetime

from simulation.simulation import Simulation
from simulation.demand_engine import preview_day
import simulation.simulation_config as config

In [6]:
import pandas as pd

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation

In [10]:
import pandas as pd

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation

sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date="2023-01-01",
    end_date="2023-01-31",
)

In [12]:
po_df = pd.DataFrame(sim.purchase_orders)

if po_df.empty:
    print("No purchase orders were created during this period.")
else:
    print(po_df["po_status"].value_counts())
    display(po_df.head())

po_status
Open        42
Received     4
Name: count, dtype: int64


,purchase_order_id,product_id,supplier_id,supplier_name,order_date,expected_receipt_date,actual_receipt_date,lead_time_days,ordered_qty,received_qty,po_status
0,PO50000,1078,S001,Fox Factory,2023-01-08,2023-01-29,2023-01-29,21,60,60,Received
1,PO50001,1068,S001,Fox Factory,2023-01-10,2023-01-31,2023-01-31,21,60,60,Received
2,PO50002,1075,S001,Fox Factory,2023-01-10,2023-01-31,2023-01-31,21,60,60,Received
3,PO50003,1114,S001,Fox Factory,2023-01-10,2023-01-31,2023-01-31,21,60,60,Received
4,PO50004,1071,S001,Fox Factory,2023-01-12,2023-02-02,None,21,61,0,Open


In [13]:
sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date="2023-01-01",
    end_date="2023-12-31",
)

orders_df = pd.DataFrame(sim.sales_orders)
lines_df = pd.DataFrame(sim.sales_order_lines)
po_df = pd.DataFrame(sim.purchase_orders)
daily_df = pd.DataFrame(sim.daily_order_summary)

print("Sales orders:", len(orders_df))
print("Sales order lines:", len(lines_df))
print("Purchase orders:", len(po_df))
print("Requested units:", lines_df["requested_qty"].sum())
print("Fulfilled units:", lines_df["fulfilled_qty"].sum())
print("Backordered units:", lines_df["backordered_qty"].sum())
print("Ending inventory:", sim.inventory["on_hand"].sum())

if not po_df.empty:
    print("Open PO units:", po_df.loc[po_df["po_status"] == "Open", "ordered_qty"].sum())
    print("Received PO units:", po_df.loc[po_df["po_status"] == "Received", "received_qty"].sum())

Sales orders: 9233
Sales order lines: 25311
Purchase orders: 746
Requested units: 50840
Fulfilled units: 43928
Backordered units: 6912
Ending inventory: 8662
Open PO units: 1648
Received PO units: 44190


In [14]:
initial_inventory_units = len(sim.products) * 40

received_units = 0

if not po_df.empty:
    received_units = po_df.loc[
        po_df["po_status"] == "Received",
        "received_qty"
    ].sum()

fulfilled_units = lines_df["fulfilled_qty"].sum()
ending_inventory_units = sim.inventory["on_hand"].sum()

inventory_difference = (
    initial_inventory_units
    + received_units
    - fulfilled_units
    - ending_inventory_units
)

inventory_difference

0

In [15]:
sim.export_tables()

In [16]:
from simulation.forecasting import (
    generate_baseline_forecast,
    calculate_forecast_metrics,
)

In [17]:
forecast_df = generate_baseline_forecast(
    sim,
    lookback_months=3,
)

forecast_df.head()

,forecast_month,product_id,model_name,category,lookback_months,forecast_qty,actual_qty,forecast_error,absolute_error,absolute_percentage_error,seasonality_adjustment
0,2023-04-01,1001,Catalina,Cross Country,3,11,5,-6,6,1.200000,1.714
1,2023-05-01,1001,Catalina,Cross Country,3,11,6,-5,5,0.833333,1.473
2,2023-06-01,1001,Catalina,Cross Country,3,6,9,3,3,0.333333,1.130
3,2023-07-01,1001,Catalina,Cross Country,3,6,1,-5,5,5.000000,0.896
4,2023-08-01,1001,Catalina,Cross Country,3,4,9,5,5,0.555556,0.789


In [18]:
calculate_forecast_metrics(forecast_df)

{'forecast_rows': 1890,
 'total_actual_qty': 41402,
 'total_forecast_qty': 41239,
 'total_absolute_error': 12055,
 'wape': 0.2912,
 'bias_pct': 0.0039,
 'mean_absolute_percentage_error': 0.553}

In [19]:
forecast_df.sort_values(
    "absolute_error",
    ascending=False,
).head(10)

,forecast_month,product_id,model_name,category,lookback_months,forecast_qty,actual_qty,forecast_error,absolute_error,absolute_percentage_error,seasonality_adjustment
653,2023-09-01,1073,Sabino,Trail,3,73,19,-54,54,2.842105,0.783
928,2023-05-01,1104,Romero,Aggressive Trail,3,84,129,45,45,0.348837,1.473
180,2023-04-01,1021,Catalina,Cross Country,3,59,15,-44,44,2.933333,1.714
747,2023-04-01,1084,Sabino,Trail,3,95,51,-44,44,0.862745,1.714
658,2023-05-01,1074,Sabino,Trail,3,129,89,-40,40,0.449438,1.473
955,2023-05-01,1107,Romero,Aggressive Trail,3,80,118,38,38,0.322034,1.473
1210,2023-08-01,1135,Oracle,Enduro,3,32,69,37,37,0.536232,0.789
657,2023-04-01,1074,Sabino,Trail,3,92,127,35,35,0.275591,1.714
956,2023-06-01,1107,Romero,Aggressive Trail,3,98,64,-34,34,0.531250,1.130
984,2023-07-01,1110,Romero,Aggressive Trail,3,55,89,34,34,0.382022,0.896


In [20]:
forecast_by_model = (
    forecast_df
    .groupby("model_name")
    .agg(
        actual_qty=("actual_qty", "sum"),
        forecast_qty=("forecast_qty", "sum"),
        absolute_error=("absolute_error", "sum"),
        forecast_error=("forecast_error", "sum"),
    )
    .reset_index()
)

forecast_by_model["wape"] = (
    forecast_by_model["absolute_error"]
    / forecast_by_model["actual_qty"]
)

forecast_by_model["bias_pct"] = (
    forecast_by_model["forecast_error"]
    / forecast_by_model["actual_qty"]
)

forecast_by_model.sort_values("wape", ascending=False)

,model_name,actual_qty,forecast_qty,absolute_error,forecast_error,wape,bias_pct
6,Sonoita,1780,1702,1034,78,0.580899,0.043820
5,Sky Island,2370,2378,1138,-8,0.480169,-0.003376
2,Rincon,3277,3284,1365,-7,0.416540,-0.002136
0,Catalina,5758,5772,1762,-14,0.306009,-0.002431
1,Oracle,7041,7025,2100,16,0.298253,0.002272
3,Romero,10317,10128,2345,189,0.227295,0.018319
4,Sabino,10859,10950,2311,-91,0.212819,-0.008380


In [21]:
sim.export_tables()

In [5]:
import pandas as pd

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation
from simulation.forecasting import generate_baseline_forecast
from simulation.analytics import export_analytics_tables

In [7]:
sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date="2021-01-01",
    end_date="2025-12-31",
)

forecast_df = generate_baseline_forecast(
    sim,
    lookback_months=3,
)

analytics_tables = export_analytics_tables(sim)

In [8]:
analytics_tables.keys()

dict_keys(['monthly_sales_summary', 'model_performance_summary', 'inventory_kpi_summary', 'forecast_accuracy_by_model', 'supplier_performance_summary', 'daily_kpi_summary'])

In [9]:
analytics_tables["monthly_sales_summary"].head()

,month_start,sales_channel,order_count,customer_count,booked_revenue,order_lines,requested_units,fulfilled_units,backordered_units,fulfilled_revenue,booked_gross_profit,fulfilled_gross_profit,service_level,backorder_rate,booked_gross_margin_pct,fulfilled_gross_margin_pct
0,2022-01-01,DTC,195,1,1002481.00,210,219,216,3,987484.00,433862.81,427472.09,0.986301,0.013699,0.432789,0.432890
1,2022-01-01,Dealer,264,34,7080648.66,1048,2334,2309,25,6999812.28,470235.28,464344.42,0.989289,0.010711,0.066411,0.066337
2,2022-02-01,DTC,210,1,1090566.00,220,234,214,20,1003286.00,469850.65,431264.54,0.914530,0.085470,0.430832,0.429852
3,2022-02-01,Dealer,289,34,7951322.68,1182,2615,2414,201,7354785.71,499302.30,459538.04,0.923136,0.076864,0.062795,0.062481
4,2022-03-01,DTC,311,1,1776137.00,340,363,314,49,1536186.00,748962.62,646331.68,0.865014,0.134986,0.421681,0.420738


In [10]:
analytics_tables["model_performance_summary"].head()

,model_name,category,sales_orders,order_lines,unique_skus,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,booked_gross_profit,fulfilled_gross_profit,service_level,backorder_rate,booked_gross_margin_pct,fulfilled_gross_margin_pct,average_selling_price
0,Romero,Aggressive Trail,16203,25089,30,51613,42035,9578,1.715949e+08,1.396967e+08,15960046.17,12943534.38,0.814427,0.185573,0.093010,0.092655,3324.645315
1,Sabino,Trail,18202,27377,30,54391,43932,10459,1.640263e+08,1.323868e+08,23945567.31,19242583.92,0.807707,0.192293,0.145986,0.145351,3015.688705
2,Oracle,Enduro,12408,17160,30,35759,32535,3224,1.411246e+08,1.283495e+08,8122611.69,7338810.13,0.909841,0.090159,0.057556,0.057178,3946.548737
3,Catalina,Cross Country,10709,14773,30,28566,24962,3604,6.280795e+07,5.486534e+07,12861151.50,11220030.96,0.873836,0.126164,0.204769,0.204501,2198.695845
4,Sky Island,eMTB,4430,6138,30,11986,11938,48,5.876006e+07,5.852482e+07,1234935.32,1230060.97,0.995995,0.004005,0.021017,0.021018,4902.391480


In [11]:
analytics_tables["inventory_kpi_summary"].head()

,product_id,sku,model_name,category,size,color,average_on_hand,average_available,minimum_available,maximum_available,ending_on_hand,ending_available,snapshot_days,stockout_days,stockout_rate
0,1106,SON-ROM-C-M-GR,Romero,Aggressive Trail,M,Granite,22.791923,22.791923,0,66,56,56,1461,416,0.284736
1,1074,SON-SAB-C-M-SG,Sabino,Trail,M,Saguaro,22.945927,22.945927,0,68,47,47,1461,416,0.284736
2,1073,SON-SAB-C-M-IR,Sabino,Trail,M,Ironwood,22.021903,22.021903,0,70,5,5,1461,413,0.282683
3,1103,SON-ROM-C-M-IR,Romero,Aggressive Trail,M,Ironwood,23.096509,23.096509,0,72,60,60,1461,405,0.277207
4,1076,SON-SAB-C-M-GR,Sabino,Trail,M,Granite,22.658453,22.658453,0,68,35,35,1461,403,0.275838


In [12]:
analytics_tables["forecast_accuracy_by_model"].head()

,model_name,category,forecast_rows,actual_qty,forecast_qty,absolute_error,forecast_error,wape,bias_pct,forecast_accuracy
0,Sonoita,Gravel,1350,7223,7168,4265,55,0.590475,0.007615,0.409525
1,Sky Island,eMTB,1350,11465,11423,5604,42,0.488792,0.003663,0.511208
2,Rincon,Downcountry,1350,15837,15550,6531,287,0.412389,0.018122,0.587611
3,Catalina,Cross Country,1350,27315,27072,8591,243,0.314516,0.008896,0.685484
4,Oracle,Enduro,1350,34125,34056,9411,69,0.275780,0.002022,0.724220


In [13]:
analytics_tables["supplier_performance_summary"].head()

,supplier_id,supplier_name,purchase_orders,open_purchase_orders,received_purchase_orders,ordered_units,received_units,open_units,average_lead_time_days,receipt_rate
0,S001,Fox Factory,2141,15,2126,131963,131038,925,21.0,0.992990
1,S002,SRAM,805,10,795,49261,48647,614,30.0,0.987536


In [14]:
analytics_tables["daily_kpi_summary"].head()

,date,weather,dealer_orders,dtc_orders,total_orders,purchase_orders_received,purchase_units_received,purchase_orders_created,purchase_units_ordered,total_on_hand_units,...,out_of_stock_skus,sales_orders,customer_count,booked_revenue,requested_units,fulfilled_units,backordered_units,fulfilled_revenue,service_level,backorder_rate
0,2022-01-01,Sunny,3,7,10,0,0,0,0,8363,...,0,10,4,124581.20,37,37,0,124581.20,1.0,0.0
1,2022-01-02,Sunny,1,7,8,0,0,0,0,8351,...,0,8,2,50094.75,12,12,0,50094.75,1.0,0.0
2,2022-01-03,Sunny,13,5,18,0,0,0,0,8225,...,0,18,11,378288.72,126,126,0,378288.72,1.0,0.0
3,2022-01-04,Sunny,14,5,19,0,0,0,0,8092,...,0,19,13,429831.45,133,133,0,429831.45,1.0,0.0
4,2022-01-05,Sunny,11,6,17,0,0,0,0,8001,...,0,17,10,307980.51,91,91,0,307980.51,1.0,0.0


In [15]:
orders_df = pd.DataFrame(sim.sales_orders)
lines_df = pd.DataFrame(sim.sales_order_lines)

monthly_revenue = analytics_tables["monthly_sales_summary"]["booked_revenue"].sum()
raw_revenue = orders_df["order_total"].sum()

round(monthly_revenue - raw_revenue, 2)

0.0

In [16]:
model_units = analytics_tables["model_performance_summary"]["requested_units"].sum()
raw_units = lines_df["requested_qty"].sum()

model_units - raw_units

0

In [17]:
sim.export_tables()

In [18]:
from simulation.database_builder import build_sqlite_database

database_result = build_sqlite_database()

database_result["database_path"]

PosixPath('/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/database/sonoran_cycles.db')

In [19]:
database_result["load_results"]

,table_name,source_csv,row_count
0,products,data/products.csv,210
1,customers,data/customers.csv,36
2,suppliers,data/suppliers.csv,10
3,calendar,data/calendar.csv,1826
4,sales_orders,outputs/sales_orders.csv,37000
5,sales_order_lines,outputs/sales_order_lines.csv,102566
6,purchase_orders,outputs/purchase_orders.csv,2946
7,inventory_history,outputs/inventory_history.csv,306810
8,forecast_history,outputs/forecast_history.csv,9450
9,daily_order_summary,outputs/daily_order_summary.csv,1461


In [20]:
database_result["table_counts"]

,table_name,row_count
0,calendar,1826
1,customers,36
2,daily_kpi_summary,1461
3,daily_order_summary,1461
4,forecast_accuracy_by_model,7
5,forecast_history,9450
6,inventory_history,306810
7,inventory_kpi_summary,210
8,model_performance_summary,7
9,monthly_sales_summary,96


In [21]:
import sqlite3
import pandas as pd

db_path = database_result["database_path"]

conn = sqlite3.connect(db_path)

In [22]:
query = """
SELECT
    model_name,
    category,
    SUM(requested_qty) AS requested_units,
    ROUND(SUM(extended_price), 2) AS booked_revenue,
    ROUND(SUM(fulfilled_revenue), 2) AS fulfilled_revenue,
    SUM(backordered_qty) AS backordered_units,
    ROUND(
        CAST(SUM(fulfilled_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS service_level
FROM sales_order_lines
GROUP BY
    model_name,
    category
ORDER BY
    booked_revenue DESC;
"""

pd.read_sql_query(query, conn)

,model_name,category,requested_units,booked_revenue,fulfilled_revenue,backordered_units,service_level
0,Romero,Aggressive Trail,51613,1.715949e+08,1.396967e+08,9578,0.814
1,Sabino,Trail,54391,1.640263e+08,1.323868e+08,10459,0.808
2,Oracle,Enduro,35759,1.411246e+08,1.283495e+08,3224,0.910
3,Catalina,Cross Country,28566,6.280795e+07,5.486534e+07,3604,0.874
4,Sky Island,eMTB,11986,5.876006e+07,5.852482e+07,48,0.996
5,Rincon,Downcountry,16454,4.181898e+07,3.987979e+07,769,0.953
6,Sonoita,Gravel,7605,1.855156e+07,1.850667e+07,18,0.998


In [23]:
query = """
SELECT
    c.region,
    c.state,
    so.sales_channel,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    SUM(sol.requested_qty) AS requested_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue
FROM sales_orders so
JOIN customers c
    ON so.customer_id = c.customer_id
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
GROUP BY
    c.region,
    c.state,
    so.sales_channel
ORDER BY
    booked_revenue DESC;
"""

pd.read_sql_query(query, conn).head(20)

,region,state,sales_channel,sales_orders,requested_units,booked_revenue,fulfilled_revenue
0,Southwest,Arizona,Dealer,4170,36586,1.151082e+08,1.004714e+08
1,West Coast,California,Dealer,3453,30322,9.685503e+07,8.628219e+07
2,National,Online,DTC,15542,17640,8.367206e+07,7.260315e+07
3,Mountain,Colorado,Dealer,2880,25451,7.500397e+07,6.472288e+07
4,West Coast,Washington,Dealer,2125,18573,5.016876e+07,4.397790e+07
5,Southeast,North Carolina,Dealer,1796,15522,4.833140e+07,4.126399e+07
6,Mountain,Utah,Dealer,745,6727,2.068213e+07,1.729577e+07
7,Midwest,Minnesota,Dealer,694,6134,1.891537e+07,1.563600e+07
8,Southeast,Tennessee,Dealer,696,6006,1.842664e+07,1.563009e+07
9,Northeast,Vermont,Dealer,714,6378,1.541408e+07,1.374442e+07


In [24]:
conn.close()